## Prompt Templates VS Messages

**Message-Based Prompting**
Message arrays are the foundation of agent systems in LangChain. When you work with agents, you'll use message arrays as input and output.

**Template-Based Prompting**
Templates allow you to create reusable, maintainable prompts with variables. 


| Approach | Use For |
|----------|---------|
| **Messages** | Agents, dynamic workflows, multi-step reasoning, tool integration |
| **Templates** | Reusable prompts, variable substitution, consistency, RAG systems |

**Both approaches are valuable**: Messages for dynamic workflows, templates for reusability and consistency.



In [10]:

import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, FewShotChatMessagePromptTemplate

load_dotenv()

True

In [3]:
model = ChatOpenAI(model="gpt-5-nano")

#### 1: Messages (Dynamic - great for agents)

In [4]:
messages = [
    SystemMessage(content="You are a helpful geography teacher."),
    HumanMessage(content="Give 2 countries that border with India"),
]

message_response = model.invoke(messages)
print("Response:", message_response.content)

Response: Pakistan and China.


### 2: Templates (Reusable - great for RAG)
**How it Helps**:
* Created with ChatPromptTemplate.from_messages()
* Uses variables like {text} and {language}
* Piped to model using | operator: template | model
* Valuable for reusability and consistency in RAG systems

#### Define Templates From Messages.

#### Re-Use defined Template multiple time with different parameters.

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful geography teacher."),
    ("human", "Give '{num}' countries that border with '{country}'"),
])

template_chain = template | model

In [5]:
template_response = template_chain.invoke({
    "num": "2",
    "country": "India",
})

print("Response:", template_response.content)


Response: Pakistan and China.


In [6]:
template_response = template_chain.invoke({
    "num": "2",
    "country": "UAE",
})

print("Response:", template_response.content)

Response: Oman and Saudi Arabia.


### Template Formats
LangChain supports different template formats. Compare ChatPromptTemplate (for chat models) with PromptTemplate (for single prompts)

### How It Works

| Template Type | Use Case | Syntax |
|--------------|----------|--------|
| **ChatPromptTemplate** | Multi-turn conversations with roles | `from_messages([("role", "content")])` |
| **PromptTemplate** | Simple single prompts | `from_template("text with {variables}")` |


#### ChatPromptTemplate - for multi-turn conversations

In [7]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful {role}."),
    ("human", "{question}"),
])

chat_chain = chat_template | model
result1 = chat_chain.invoke({"role": "math tutor", "question": "What is 2+2?"})
print("ChatPromptTemplate:", result1.content)

ChatPromptTemplate: 4

Two plus two equals four. If you’d like, I can show how this works in different number bases.


#### PromptTemplate - for simple single prompts

In [9]:

simple_template = PromptTemplate.from_template(
    "Answer this {topic} question briefly: {question}"
)

simple_chain = simple_template | model
result2 = simple_chain.invoke({"topic": "Geography", "question": "What is capitak of Japan?"})
print("PromptTemplate:", result2.content)

PromptTemplate: Tokyo.


## Few-Shot Prompting with Templates
**How It Works**
* **Define Examples**: Create input/output pairs that demonstrate the pattern
* **Example Template**: Define how each example should be formatted
* **Few-Shot Template**: Wrap examples in `FewShotChatMessagePromptTemplate`
* **Combine**: Add system message, examples, and user input together


In [11]:
# Teaching examples
examples = [
    {"input": "happy", "output": "😊"},
    {"input": "sad", "output": "😢"},
    {"input": "excited", "output": "🎉"},
]

# Example template having placeholders for input and outputs. 
example_template = ChatPromptTemplate.from_messages([
    ("user", "{input}"),
    ("assistant", "{output}"),
])

#### Few-shot template

In [12]:
few_shot_template = FewShotChatMessagePromptTemplate(
    example_prompt=example_template,
    examples=examples,
)

#### Final template combining system message + examples + user input

In [13]:
final_template = ChatPromptTemplate.from_messages([
    ("system", "You are an emoji translator. Convert words to emojis."),
    few_shot_template,
    ("user", "{input}"),
])

chain = final_template | model

#### Test with new inputs

In [14]:
for word in ["angry", "love", "confused"]:
    result = chain.invoke({"input": word})
    print(f"{word}: {result.content}")

angry: 😡
love: ❤️
confused: 😕


## Template Composition
* Create Base Template, which can further be used/appended in further templates for different usecases
* e.g., First create a `base_instructions` which can be generic within organization.
* Then while creating templates for specific use-case `educator_template`, that can include `base_instructions` plus adding more instructions specific to usecase.

#### Base template with common elements

In [17]:
# Base template with common elements
base_instructions = """
    You are a {role} assistant.
    Your communication style is {style}.
    Always be helpful and professional.
"""

#### Compose templates for different use cases

In [19]:
educator_template = ChatPromptTemplate.from_messages([
    ("system", base_instructions),
    ("system", "Focus on teaching concepts clearly with examples."),
    ("human", "{question}"),
])


In [20]:
# Use the educator template
educator_chain = educator_template | model

result = educator_chain.invoke({
    "role": "Python programming",
    "style": "friendly and encouraging",
    "question": "What is a list comprehension?",
})

In [22]:
print(result.content)

A list comprehension is a compact way to create a new list by applying an expression to each item in an existing iterable, with an optional filter. It’s a concise, readable replacement for a for-loop that builds a list.

Syntax
- Basic: [expression for item in iterable]
- With a filter: [expression for item in iterable if condition]
- With multiple loops: [expression for a in A for b in B]

Examples
- Simple mapping (square numbers 0–4):
  - [x*x for x in range(5)]
  - Result: [0, 1, 4, 9, 16]

- Filtering (even numbers 0–19):
  - [n for n in range(20) if n % 2 == 0]
  - Result: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

- Nested loops (pairs):
  - [(i, j) for i in range(3) for j in range(2)]
  - Result: [(0,0), (0,1), (1,0), (1,1), (2,0), (2,1)]

- Conditional expression inside (negate odd numbers):
  - [x if x % 2 == 0 else -x for x in range(5)]
  - Result: [0, -1, 2, -3, 4]

How it relates to a for-loop
- Equivalent expansion:
  - result = []
  - for item in iterable:
      if condition (